## Project Overview

The core idea of this project is to prove that satellite images of residential areas can provide critical, predictive information when estimating house prices. Specifically, our work focuses on extracting the density of the road network (the number of pixels corresponding to roads) from satellite imagery. Existing literature suggests a non-linear relationship here: too few roads indicate extreme isolation (lowering the price), while too many roads indicate heavy traffic or industrial areas (also lowering the price). Optimal road access is expected to correlate with higher property values.

To achieve this, we rely on the **DeepGlobe 2018 Road Extraction Dataset**:

> @InProceedings{DeepGlobe18,
>  author = {Demir, Ilke and Koperski, Krzysztof and Lindenbaum, David and Pang, Guan and Huang, Jing and Basu, Saikat and Hughes, Forest and Tuia, Devis and Raskar, Ramesh},
>  title = {DeepGlobe 2018: A Challenge to Parse the Earth Through Satellite Images},
>  booktitle = {The IEEE Conference on Computer Vision and Pattern Recognition (CVPR) Workshops},
>  month = {June},
>  year = {2018}
> }

Using this dataset, we train a semantic segmentation model capable of accurately detecting and masking roads from an aerial perspective. 

Next, we process a tabular dataset of California houses, which includes property characteristics, prices, and exact geographic coordinates. The final data gathering step is to extract the corresponding high-resolution satellite images centered on each property using the **Google Earth Engine API**.

---

## Format of the Project

### 1. Data Acquisition & Preprocessing
* Load the California housing dataset containing the target variable (median house value) and tabular features (total rooms, age, median income, etc.).
* Automate the downloading of satellite images via Google Earth Engine, ensuring each image is perfectly centered on the specific latitude and longitude of the respective house.

### 2. Semantic Segmentation Model Development
* Train a deep learning vision model (e.g., U-Net) using the DeepGlobe 2018 dataset.
* Validate the model to ensure it accurately identifies and segments road pixels in diverse aerial landscapes.

### 3. Feature Extraction & Dataset Merging
* Feed the newly downloaded California house satellite images into the trained segmentation model.
* Calculate the total number (or percentage) of road pixels within the bounding box of each property. 
* Append this calculated metric as a new spatial feature (e.g., `road_pixel_count`) into the original California housing DataFrame.

### 4. Baseline Model Training (Tabular Data Only)
* Train a standard regression model (e.g., Random Forest, XGBoost, or a Multi-Layer Perceptron) to predict the house price using **only** the traditional house characteristics.

### 5. Enhanced Model Training (Multimodal Data)
* Train the exact same regression model architecture, but this time include the newly extracted spatial feature (`road_pixel_count`) alongside the traditional tabular data.

### 6. Evaluation and Conclusion
* Compare the predictive performance of both models using standard regression metrics such as RMSE (Root Mean Squared Error) and $R^2$. 
* Determine if the enhanced model significantly outperforms the baseline, thereby proving the hypothesis that computer vision applied to satellite context adds tangible predictive value to real estate pricing models.

LOAD THE CALIFORNIA HOUSES DATASET:

In [1]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("hosammhmdali/house-price-dataset")
print("Path to dataset files:", path)

# Buscamos el archivo CSV dentro de la ruta de descarga
# (Suele llamarse data.csv, house_prices.csv, etc. Puedes mirar el nombre con la Opción 1)
archivo_csv = os.path.join(path, "housing.csv") 

# Cargamos el dataset
df = pd.read_csv(archivo_csv)

# Mostramos las primeras filas
df.head()

Path to dataset files: /Users/gal.lagelpibuxade/.cache/kagglehub/datasets/hosammhmdali/house-price-dataset/versions/1


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
import ee
import os
import requests
import pandas as pd

# 1. Inicializar com o teu projeto
ee.Initialize(project="pklot-termposter") 

# 2. Pasta de saída
OUTPUT_DIR = "imagenes_satelite"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Config
START_DATE = "2010-01-01"        # busca NAIP desde esta data até hoje
BUFFER_METROS = 100              # ~200m x 200m ao redor do ponto
DIMENSIONS = "1024x1024"         # resolução da imagem em pixels
FORCE_REDOWNLOAD = False         # True para baixar de novo mesmo se já existir
SOLO_MAS_RECIENTE = True         # True = só a imagem mais recente por ponto
                                 # False = todas as datas disponíveis

# 4. ID único e coluna de rutas
df['id_casa'] = range(1, len(df) + 1)
df['ruta_imagen'] = ""

# 5. Função de download usando NAIP
def descargar_naip(lat, lon, id_casa):
    punto = ee.Geometry.Point([lon, lat])
    region = punto.buffer(BUFFER_METROS).bounds()

    coleccion = (ee.ImageCollection("USDA/NAIP/DOQQ")
                 .filterBounds(punto)
                 .filterDate(START_DATE, "2026-01-01")
                 .sort("system:time_start"))

    n = coleccion.size().getInfo()
    if n == 0:
        print(f"  ⚠️  Sin imágenes NAIP para casa ID {id_casa} (¿fuera de EE.UU.?)")
        return []

    # Decide quais imagens baixar
    if SOLO_MAS_RECIENTE:
        imagenes = [ee.Image(coleccion.sort("system:time_start", False).first())]
    else:
        lista = coleccion.toList(n)
        imagenes = [ee.Image(lista.get(i)) for i in range(n)]

    rutas = []
    for img in imagenes:
        date = img.date().format("YYYY-MM-dd").getInfo()
        nombre_archivo = os.path.join(OUTPUT_DIR, f"casa_{id_casa}_{date}.png")

        if os.path.exists(nombre_archivo) and not FORCE_REDOWNLOAD:
            print(f"  {date} - ya existe, saltando")
            rutas.append(nombre_archivo)
            continue

        url = img.select(["R", "G", "B"]).getThumbURL({
            "region": region,
            "dimensions": DIMENSIONS,
            "format": "png"
        })

        r = requests.get(url)
        if r.status_code == 200:
            with open(nombre_archivo, "wb") as f:
                f.write(r.content)
            print(f"  📸 Casa ID {id_casa} ({date}) → {nombre_archivo}")
            rutas.append(nombre_archivo)
        else:
            print(f"  ❌ Error {r.status_code} en casa ID {id_casa} ({date})")

    return rutas

# 6. Descargar las primeras 5 como prueba
print("Iniciando descarga con NAIP...\n")
for indice, fila in df.head(500).iterrows():
    id_actual = fila['id_casa']
    print(f"=== Casa ID {id_actual} ===")
    rutas = descargar_naip(fila['latitude'], fila['longitude'], id_casa=id_actual)

    if rutas:
        # Guarda la ruta más reciente (o la única) en el DataFrame
        df.at[indice, 'ruta_imagen'] = rutas[-1]

print("\n¡Proceso terminado!")
df[['id_casa', 'latitude', 'longitude', 'median_house_value', 'ruta_imagen']].head()

/Users/gal.lagelpibuxade/Desktop/Application-of-Deep-learning-image/.venv/lib/python3.13/site-packages/ee/main.py:150: SyntaxWarning: invalid escape sequence '\d'
  lines = filter(lambda x: re.match("^\d+ bytes", x), data.splitlines())
/Users/gal.lagelpibuxade/Desktop/Application-of-Deep-learning-image/.venv/lib/python3.13/site-packages/ee/main.py:150: SyntaxWarning: invalid escape sequence '\d'
  lines = filter(lambda x: re.match("^\d+ bytes", x), data.splitlines())


ModuleNotFoundError: No module named 'StringIO'

Vamos ahora a entrenar un modelo que aprenda a detectar el numero de pixeles de una imagen que son una carretera. Primero descargamos el dataset de entrenamiento.

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("balraj98/deepglobe-road-extraction-dataset")

print("Path to dataset files:", path)

archivo_csv_fotos = os.path.join(path, "housing.csv") 

# Cargamos el dataset
df_fotos = pd.read_csv(archivo_csv)

# Mostramos las primeras filas
df.head()

  4%|▎         | 144M/3.79G [04:09<1:47:57, 606kB/s]  


KeyboardInterrupt: 